# 10 | Decision synthesis and presentation evidence

**Author: Chanakya**

Connect the analyses into the four requested decisions. Recommendations are starting treatments with explicit reversal conditions. This notebook does not claim a completed slide deck or a measured optimal strategy.

In [ ]:
from pathlib import Path
import sys
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'data/manifests/release.json').exists())
sys.path.insert(0, str(ROOT))
from src.analysis_common import *
from datetime import datetime, timedelta
from zoneinfo import ZoneInfo
rng = np.random.default_rng(CFG['seed'])
print('Offline inputs:', CFG['raw_release'], '| Author: Chanakya')
import src.analysis_common as shared
shared.ACTIVE_NOTEBOOK='10_strategy_and_claims'
shared.ACTIVE_SOURCES=[]

## 1. Sport × entitlement segmentation
Sport interests overlap. Ownership states determine whether the action is acquisition, upgrade or retention. No row has an invented audience size. Require a tennis-interest signal and a usable occasion before cross-sport promotion.

In [ ]:
sports={'F1':'Rivalry and championship story, outside selected race coverage','Football':'Specific club or competition break, concise tennis matchup introduction','MotoGP':'Player rivalry and season progression, outside selected race coverage','Tennis-first comparator':'Confirmed player/event intent and clear timing'}
states={'Never paid / lapsed, ATP uncovered':'Separate first-ever and reactivated users. Cheapest covering entry offer, no automatic annual push.','Active payer, ATP uncovered':'Incremental coverage for chosen basket. Compare tournament, season and portfolio options.','ATP already covered':'Suppress redundant sale. Surface included live/replay occasion, evaluate renewal contribution.'}
segments=[]
for sport,bridge in sports.items():
 for ownership,action in states.items():segments.append(dict(sport=sport,ownership=ownership,observable_eligibility='Verified entitlement + explicit or observed tennis interest + usable time',message_hypothesis=bridge,lead_action=action,success='Incremental new payer' if ownership.startswith('Never') else ('Portfolio contribution from additional coverage' if ownership.startswith('Active') else 'Incremental renewal contribution'),kill_rule='No positive incremental portfolio contribution, insufficient relevant reach, or playback/refund harm'))
segments=pd.DataFrame(segments);display(table(segments,'10_segmentation_matrix'))
# Structured decision rules make conditions visible instead of an opaque score.
rules=pd.DataFrame([
('Already covered','Included next occasion','Incremental renewal contribution > message and service cost','Do not sell redundant access'),
('One intended tennis event','Tournament access','Positive incremental cohort contribution','Stop paid media above allowable acquisition cost'),
('Repeated tennis, enough remaining events','Minimum-cost season or monthly coverage','Basket price advantage and scope verified','Switch if monthly covers the basket more cheaply'),
('Multiple sports across dates','Exact minimum-cost portfolio coverage','Fits dates and devices, increment over existing ownership known','Do not assume annual is best'),
('No usable live time','Existing replay or feasibility-tested rental','Incremental contribution exceeds displaced live purchases','Reject rental if cannibalization dominates'),
('Player-led intent','Confirmed event / flexible player bundle','Participation and refund terms clear','No unconditional promise of a semifinal/final'),
('Low intent / no usable occasion','Suppress or very small exploration cell','Demonstrable persuasion and reachable scale','Do not spend against registration count')],columns=['customer_state','lead_recommendation','economic_gate','reversal']);display(table(rules,'10_recommendation_rules'))

## 2. Export computed claims, including their limits
Each claim links to a table, notebook and intended slide. A scenario output remains a scenario in the slide headline. Do not copy an external audience estimate into the conversion denominator.

In [ ]:
timing=pd.read_csv(ROOT/'outputs/tables/01_timing_robustness.csv');econ=pd.read_csv(ROOT/'outputs/tables/07_contribution_grid.csv');search=pd.read_csv(ROOT/'outputs/tables/05_search_associations.csv');reviews=pd.read_csv(ROOT/'outputs/tables/04_store_coverage.csv');blended=pd.read_csv(ROOT/'outputs/tables/08_blended_programme_cac.csv');power=pd.read_csv(ROOT/'outputs/tables/09_binary_power_grid.csv')
season=econ[(econ.price==399)&(econ.tax==.18)&(econ.other_variable_cost==30)&(econ.cac==175)].iloc[0]
claims=[
('C01','Robust final-start fit across all chosen timing cells',int((timing.fraction_of_design_cells==1).sum()),'of 12 purposively selected finals','01_timing_robustness.csv','01','3','Design grid, not probability or full-season share'),
('C02','Explicit tennis mentions in collected mobile and iOS reviews',int(reviews.tennis_mentions.sum()),'mentions','04_store_coverage.csv','04','1','Sparse keyword signal, not a tennis customer survey'),
('C03','F1/FanCode rank correlation in common request',float(search[search.sport=='Formula 1'].level_rho.iloc[0]),'Spearman rho','05_search_associations.csv','05','1','Search association, not paid acquisition attribution'),
('C04','Season contribution before marketing under selected cost case',float(season.pre_marketing_contribution),'INR','07_contribution_grid.csv','07','5','18% inclusive tax, 2% fee plus fee tax, INR30 additional cost'),
('C05','Season contribution after INR175 acquisition in same case',float(season.after_acquisition),'INR','07_contribution_grid.csv','07','5','Rights excluded, not observed company margin'),
('C06','Incrementality needed at attributed INR175 for INR200 iCAC',.875,'fraction','07_incrementality_cac.csv','07','5','Algebraic target requirement, not measured fraction'),
('C07','Acquisition-cell ceiling after 10% reserve at INR200 target',180,'INR','08_measurement_reserve_sensitivity.csv','08','7','Reserve funds no separately credited payers'),
('C08','Working-test programme CAC',float(blended[blended.scenario=='Working test'].fully_loaded_icac.iloc[0]),'INR','08_blended_programme_cac.csv','08','7','Conditional on unmeasured response assumptions, not forecast'),
('C09','Users for 1% baseline and 20% relative lift',int(power[np.isclose(power.baseline,.01)&np.isclose(power.relative_lift,.2)].total_users.iloc[0]),'total randomized users','09_binary_power_grid.csv','09','8','Two-sided 5% alpha, 80% power, independent equally sized arms')]
claims=pd.DataFrame(claims,columns=['claim_id','claim','value','unit','table','notebook','planned_slide','qualification']);table(claims,'10_claim_register');(ROOT/'outputs/reports/claim_register.json').write_text(claims.to_json(orient='records',indent=2));display(claims)
report('10_decision_summary',f"The strongest case is profitable viewing occasions plus the next relevant reason to return. In the declared sensitivity grid, {int((timing.fraction_of_design_cells==1).sum())} of 12 selected final starts fit every window/delay cell. This is not a population result. Only {int(reviews.tennis_mentions.sum())} explicit tennis-related review mentions were found. Selected season contribution is INR{season.pre_marketing_contribution:.2f} before acquisition and INR{season.after_acquisition:.2f} after INR175, under stated costs. The working channel scenario implies INR{float(blended[blended.scenario=='Working test'].fully_loaded_icac.iloc[0]):.2f} fully loaded iCAC, so the chosen allocation must earn expansion through tests rather than be sold as already feasible.")

## 3. A connected eight-slide argument
The executive summary is a separate mandatory page, written from the verified claims. Source and scenario notes remain on the main slides where they affect decisions. Core answers are not hidden in an appendix.

In [ ]:
story=[
(1,'Diagnose profitable ATP growth','Case denominator audit + search/review limits','Separate interest, access and incremental payment','00,04,05','00_evidence_coverage','Which customers require which intervention?'),
(2,'Target audience states, not undeduplicated sport totals','F1/football/MotoGP × entitlement matrix','Select messages and outcomes per state','10','10_segmentation_matrix.csv','Their usable occasions define the inventory that matters.'),
(3,'Match the offer to a usable occasion','Balanced final-start sample + fixture clashes','Prefer verified local timing and conflict suppression','01,02','01_final_start_times','Those occasions define an explicit coverage basket.'),
(4,'Offer the cheapest suitable coverage','Pass comparison, dynamic rules, rental/player safeguards','Conditional switch map with monthly scope fork','03,06','03_basket_switch_map','Coverage choices determine cashflows and leakage.'),
(5,'Set contribution limits before buying growth','Tax/fee bridge, finite repeat and credit hurdles','Allowable CAC and discount rejection frontier','07','07_contribution_bridge','Required future value defines the retention task.'),
(6,'Earn the next relevant paid relationship','Dated event journeys and pay-to-play instrumentation','Separate engagement, payment and genuine renewal','04,09','09_next_event_journeys.csv','Treatments need capacity and funded channels.'),
(7,'Fund a measured learning allocation','40/20/20/10/10 with cost builds and conditional CAC','Reconcile total spend and reserve-adjusted ceilings','08','08_channel_cac','Response uncertainty defines a staged pilot.'),
(8,'Pilot, stop or scale','Power and contribution measurement','Owners, maturity and nonpositive-value kill rules','09,10','09_power_requirements','Return to the objective: profitable incremental relationships.')]
story=pd.DataFrame(story,columns=['slide','question','evidence','decision','notebooks','primary_artifact','bridge_to_next']);display(table(story,'10_storyboard_evidence'))
coverage=pd.read_csv(ROOT/'outputs/tables/00_requirement_map.csv');coverage['analysis_status']='Implemented with stated public-data limits';coverage['deck_status']='Not built in this phase';display(table(coverage,'10_deliverable_coverage'))
check('10_synthesis',{'named_sports_visible':{'F1','Football','MotoGP'}<=set(segments.sport),'mutually_exclusive_ownership_columns':segments.ownership.nunique()==3,'eight_substantive_slides':len(story)==8,'four_deliverables':coverage.question.nunique()==4,'claim_tables_exist':all((ROOT/'outputs/tables'/f).exists() for f in claims.table),'no_invented_segment_sizes':'segment_size' not in segments.columns})

## What remains outside these analyses
Current ATP checkout and monthly inclusion, internal rights and enterprise delivery costs, actual payer cohorts, sport overlap, WTP and causal conversion remain unknown. No synthetic respondents or claimed experiment results are supplied. The public-data package supports a differentiated, testable strategy and explicit economic gates. It does not establish a profit forecast or a winning outcome.